####MERGE
The MERGE command in Delta Lake allows you to perform upserts (simultaneously updating existing rows and inserting new rows) along with deletes in a single, atomic transaction. It is used in Change Data Capture and Prevents Data Duplication with ACID Guarantees.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####1. Create the customers target table

In [0]:
%sql
DROP TABLE IF EXISTS customers;

CREATE OR REPLACE TABLE customers (
  customer_id  INT     COMMENT 'Primary key',
  name         STRING  COMMENT 'Full name',
  email        STRING  COMMENT 'Contact email',
  tier         STRING  COMMENT 'bronze / silver / gold',
  updated_at   TIMESTAMP COMMENT 'Last modified timestamp'
)
USING DELTA
COMMENT 'Customer master — target for MERGE demos';

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'silver', '2024-06-01 08:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'bronze', '2024-06-01 08:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'gold',   '2024-06-01 08:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze', '2024-06-01 08:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver', '2024-06-01 08:00:00');
  

####2. Create the source batch

In [0]:
from pyspark.sql import Row
from datetime import datetime

updates = [
    # customer_id 2: email changed, upgrade to silver
    Row(customer_id=2, name='Bob Patel',
        email='bob.patel@newdomain.com', tier='silver',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
    # customer_id 3: tier upgraded to platinum
    Row(customer_id=3, name='Carol Santos',
        email='carol@example.com', tier='platinum',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
    # customer_id 6: brand new customer
    Row(customer_id=6, name='Frank Osei',
        email='frank@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
]

df_updates = spark.createDataFrame(updates)
df_updates.createOrReplaceTempView("customer_updates")
print("Source batch ready — 2 updates, 1 new record.")

In [0]:
%sql
select * from customer_updates

>Three rows in the source.\
Two match existing customers — IDs 2 and 3. One is brand new — ID 6.

####3. Basic MERGE: insert-or-update

>NULL does not equal NULL in a join, so any NULL-keyed rows will always be treated as unmatched.\
 Make sure your join key is unique and non-null in both source and target.

In [0]:
%sql
MERGE INTO customers AS target          -- the table being modified
USING customer_updates AS source        -- the incoming batch
ON target.customer_id = source.customer_id  -- the join key

WHEN MATCHED THEN                       -- source row found a match in target
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN                   -- source row has no match in target
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at);

####4. Verify the result

In [0]:
%sql
SELECT * FROM customers ORDER BY customer_id;

###MERGE clause combinations

####SCD Type 1 with late arriving records
>Your pipeline receives a sync from the upstream CRM. The source always includes the full customer record — but it is not always the freshest version. Network delays, retry logic, and out-of-order delivery mean you sometimes get a record that is older than what you already have in the target.\

>If you run the basic MERGE, you would overwrite the target's newer data with the source's stale data. The fix: add a condition to the WHEN MATCHED clause — only update if the source record is actually newer.

####1. Reset customers to a clean state for this demo

In [0]:
%sql
CREATE OR REPLACE TABLE customers (
  customer_id  INT,
  name         STRING,
  email        STRING,
  tier         STRING,
  updated_at   TIMESTAMP
)
USING DELTA;

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'silver',   '2024-06-15 09:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'silver',   '2024-06-15 09:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'platinum', '2024-06-15 09:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze',   '2024-06-15 09:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver',   '2024-06-15 09:00:00');

####2. Source batch with one late-arriving record

In [0]:
from pyspark.sql import Row
from datetime import datetime

late_batch = [
    # ID 1: fresh update — should be applied
    Row(customer_id=1, name='Alice Nguyen',
        email='alice.new@example.com', tier='gold',
        updated_at=datetime(2024, 6, 20, 10, 0, 0)),
    # ID 2: stale — source timestamp is OLDER than target (late arrival)
    # target has 2024-06-15, source has 2024-05-28 — should be SKIPPED
    Row(customer_id=2, name='Bob Patel',
        email='bob.stale@example.com', tier='bronze',
        updated_at=datetime(2024, 5, 28, 6, 0, 0)),
    # ID 6: new customer — should be inserted
    Row(customer_id=6, name='Grace Lin',
        email='grace@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 20, 10, 0, 0)),
]

df_late = spark.createDataFrame(late_batch)
df_late.createOrReplaceTempView("crm_sync")
print("CRM sync batch ready.")

In [0]:
%sql
select * from crm_sync

>Three source records.\
ID 1 is fresh — June 20, newer than our target.\
ID 2 is stale — May 28, older than our target's June 15 timestamp.\
ID 6 is a new customer.\
That simulates a late-arriving retry.\
The basic MERGE would overwrite Bob with the stale data. The conditional MERGE will skip him.

####3. Conditional MERGE: SCD Type 1

In [0]:
%sql
MERGE INTO customers AS target
USING crm_sync AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at);

####4. Verify SCD Type 1 result

In [0]:
%sql
SELECT * FROM customers ORDER BY customer_id;

>Alice is gold with the new email.\
Bob is still silver with his original email — the stale source record was correctly ignored.\
Grace is a new row.\
That is SCD Type 1: update in place if fresher, skip if stale, insert if new — in one MERGE statement.

####5. Delete-on-match

>The third pattern: a full-sync source.\
Instead of a delta of changes, you receive the complete current state of the system.\
Any customer not in this source no longer exists in the upstream and should be removed from the target.

####6. Full-sync source: 5 of the 6 current customers

In [0]:
from pyspark.sql import Row
from datetime import datetime
full_sync = [
    Row(customer_id=1, name='Alice Nguyen',
        email='alice.new@example.com', tier='gold',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=2, name='Bob Patel',
        email='bob@example.com', tier='silver',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=3, name='Carol Santos',
        email='carol@example.com', tier='platinum',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=4, name='David Kim',
        email='david@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=5, name='Eva Müller',
        email='eva@example.com', tier='silver',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    # customer_id 6 (Grace) is deliberately absent — deleted upstream
]

df_full = spark.createDataFrame(full_sync)
df_full.createOrReplaceTempView("full_sync_source")

# Show current target count before MERGE
current_count = spark.table("customers").count()
print(f"Target row count before MERGE: {current_count}")

>The full-sync source has only five rows. \
Grace is absent — she was deleted from the upstream system. \
The MERGE needs to handle all three cases: update existing, insert genuinely new ones, and delete those not in the source.

In [0]:
%sql
MERGE INTO customers AS target
USING full_sync_source AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED BY TARGET THEN
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at)

WHEN NOT MATCHED BY SOURCE THEN
  DELETE;

####7. Verify: Grace is gone, others intact

In [0]:
%sql
SELECT * FROM customers ORDER BY customer_id;

####8. Performance note: partition pruning on the join key
>By default, MERGE scans the entire target table to find matches — even if the source only touches a small slice of it. \
On a 10 terabyte table, that is an expensive full scan every time the pipeline runs.


>The fix: make sure a partition key or Liquid Clustering key appears in the ON clause. When Delta sees a column it can use for file pruning in the join condition, it skips the files that cannot possibly contain a match before the join runs. A MERGE that took an hour on a full scan can drop to seconds once you add one extra column to the ON clause


>Example:
```
    MERGE INTO customers AS target
    USING source
    ON  target.region = source.region    -- partition pruning happens here
    AND target.customer_id = source.customer_id   -- then the actual join key
```



####Final Note
>The join condition is everything. The ON clause defines what "matching" means. \
If it is wrong — non-unique key, nullable columns, wrong column — the MERGE produces silent incorrect results. 

Before writing any MERGE in production: confirm the join key uniquely identifies one row in the target, and confirm it is never NULL. Those two checks prevent the majority of MERGE bugs.

>Three patterns to own:

First — Basic upsert: WHEN MATCHED UPDATE, WHEN NOT MATCHED INSERT. \
Second — Conditional SCD Type 1: WHEN MATCHED AND condition UPDATE, skipping late-arriving or stale records.\
Third — Full-sync with delete: all three clauses including WHEN NOT MATCHED BY SOURCE DELETE. The safe alternative to truncate-and-reload.

>MERGE is atomic. Every update, insert, and delete in a single MERGE statement is one Delta transaction. 

>